In [1]:
# =============================================================================
# CS611 — Dengue Outbreak Risk Prediction
# Train + Evaluate (Kaggle, no MLflow)
# =============================================================================
# Runs end-to-end:
#   1. Load gold features
#   2. Temporal split  (Train / Val[calib+thresh] / Test / OOT)
#   3. Optuna HPT — 150 trials, 5-fold time-series CV — for XGBoost & LightGBM
#      (class imbalance handled via tuned scale_pos_weight, NO resampling;
#       CV objective = F2-score, not raw recall — see note below)
#   4. Probability calibration (Platt/sigmoid) fit on a held-out Val-calib
#      slice, separate from the slice used to pick the threshold
#   5. Calibrate decision threshold per model on Val-thresh (precision-
#      maximizing, subject to recall >= RECALL_TARGET)
#   6. Threshold sensitivity check (recall/precision at thr +/- deltas on
#      Test) — flags whether the chosen threshold sits on a stable plateau
#      or a steep slope that won't transfer
#   7. Evaluate on Val / Test / OOT at the calibrated threshold (incl. PR-AUC)
#   8. PSI (score distribution shift)
#   9. SHAP top features + raw feature~label correlation (cross-check —
#      flags features SHAP ranks low that actually carry real signal, or
#      vice versa)
#  10. Plots: optuna history | PR curves | threshold sweep | metrics table
#
# Why F2 instead of recall, and why calibrate a threshold:
#   Optimizing raw recall (with scale_pos_weight free up to 10x) rewards the
#   degenerate "flag almost everything positive" solution — recall hits ~1.0
#   trivially while precision collapses. F2 (beta=2) still weights recall
#   above precision — fitting an outbreak early-warning use case where a
#   missed cluster is worse than a false alarm — but unlike raw recall it is
#   bounded by precision too, so the degenerate solution is no longer free.
#   Threshold calibration then separates "how well does the model rank risk"
#   (tuned via F2/AUC) from "what operating point do we deploy at" (chosen
#   explicitly off a PR curve, not left to an arbitrary 0.5 default).
#
# Why split Val into calib / thresh, instead of reusing all of Val twice:
#   Raw GBM probabilities under a tuned scale_pos_weight are not well
#   calibrated. Fitting the Platt scaler AND picking the threshold on the
#   exact same rows lets calibration quietly overfit the threshold choice.
#   Using disjoint temporal slices for "fit calibration" and "pick
#   threshold" keeps those two decisions independent.
# =============================================================================

import json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

warnings.filterwarnings("ignore")

# ── paths ─────────────────────────────────────────────────────────────────────
# Update INPUT_PATH to wherever your parquet lives in Kaggle
INPUT_PATH = Path("/kaggle/input/datasets/phuchungdinh/mle-data/subzone_features.parquet")
OUT_DIR    = Path("/kaggle/working")

# ── temporal boundaries ───────────────────────────────────────────────────────
TRAIN_END  = pd.Timestamp("2018-12-31")
VAL_START  = pd.Timestamp("2019-01-01");  VAL_END  = pd.Timestamp("2019-06-30")
VAL_MID    = pd.Timestamp("2019-03-31")   # Val splits into calib (Jan-Mar) / thresh (Apr-Jun)
TEST_START = pd.Timestamp("2019-07-01");  TEST_END = pd.Timestamp("2019-12-31")
OOT_START  = pd.Timestamp("2020-01-01")

# ── features / label ──────────────────────────────────────────────────────────
FEATURES = [
    "rainfall_lag1w", "rainfall_lag2w", "rainfall_lag4w",
    "cluster_count_rolling2w", "cluster_count_rolling4w",
    "recent_cases_rolling2w",  "recent_cases_rolling4w",
    "population", "elderly_pct", "area_km2",
    "population_density", "vulnerability_index",
]
LABEL = "label"

# ── hyperparameters ───────────────────────────────────────────────────────────
N_OPTUNA_TRIALS = 150
N_CV_FOLDS      = 5
RECALL_TARGET   = 0.70
F2_BETA         = 2.0   # CV objective: F-beta with beta=2 (recall weighted 2x precision)

# ── colours ───────────────────────────────────────────────────────────────────
CLR = {"xgboost": "#e67e22", "lightgbm": "#2980b9"}


# =============================================================================
# 1. DATA
# =============================================================================

def load_and_split(path: Path):
    print(f"[data] Loading {path.name}")
    df = pd.read_parquet(path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.dropna(subset=FEATURES + [LABEL])

    splits = {
        "train":      df[df["date"] <= TRAIN_END],
        "val":        df[(df["date"] >= VAL_START)  & (df["date"] <= VAL_END)],
        "val_calib":  df[(df["date"] >= VAL_START)  & (df["date"] <= VAL_MID)],
        "val_thresh": df[(df["date"] >  VAL_MID)    & (df["date"] <= VAL_END)],
        "test":       df[(df["date"] >= TEST_START) & (df["date"] <= TEST_END)],
        "oot":        df[df["date"] >= OOT_START],
    }
    for name, s in splits.items():
        pos_rate = s[LABEL].mean()
        print(f"  {name:5s}: {len(s):6,} rows | "
              f"positive: {s[LABEL].sum():,} ({pos_rate:.1%})")
    return splits


# =============================================================================
# 2. TIME-SERIES CV FOLDS
# =============================================================================

def ts_cv_folds(train_df: pd.DataFrame, n_folds: int = N_CV_FOLDS):
    """Sliding-window folds — future never leaks into past."""
    dates     = sorted(train_df["date"].unique())
    fold_size = len(dates) // (n_folds + 1)
    folds = []
    for i in range(1, n_folds + 1):
        t_end  = i * fold_size
        v_end  = t_end + fold_size
        t_idx  = dates[:t_end]
        v_idx  = dates[t_end:v_end]
        if not v_idx:
            continue
        folds.append((
            train_df[train_df["date"].isin(t_idx)],
            train_df[train_df["date"].isin(v_idx)],
        ))
    return folds


# =============================================================================
# 3. OPTUNA OBJECTIVE
# =============================================================================

def make_objective(train_df: pd.DataFrame, model_type: str):
    from sklearn.metrics import fbeta_score

    def objective(trial):
        if model_type == "xgboost":
            import xgboost as xgb
            params = dict(
                n_estimators     = trial.suggest_int("n_estimators", 100, 500),
                max_depth        = trial.suggest_int("max_depth", 3, 8),
                learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                subsample        = trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
                min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
                scale_pos_weight = trial.suggest_float("scale_pos_weight", 1.0, 10.0),
                use_label_encoder=False, eval_metric="logloss", random_state=42,
            )
            clf = xgb.XGBClassifier(**params)
        else:
            import lightgbm as lgb
            params = dict(
                n_estimators      = trial.suggest_int("n_estimators", 100, 500),
                max_depth         = trial.suggest_int("max_depth", 3, 8),
                learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                subsample         = trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree  = trial.suggest_float("colsample_bytree", 0.6, 1.0),
                min_child_samples = trial.suggest_int("min_child_samples", 5, 50),
                scale_pos_weight  = trial.suggest_float("scale_pos_weight", 1.0, 10.0),
                random_state=42, verbose=-1,
            )
            clf = lgb.LGBMClassifier(**params)

        cv_scores = []
        for fold_tr, fold_vl in ts_cv_folds(train_df):
            Xf = fold_tr[FEATURES].values;  yf = fold_tr[LABEL].values
            Xv = fold_vl[FEATURES].values;  yv = fold_vl[LABEL].values
            clf.fit(Xf, yf)
            yp = clf.predict(Xv)
            cv_scores.append(fbeta_score(yv, yp, beta=F2_BETA, zero_division=0))

        return float(np.mean(cv_scores))

    return objective


# =============================================================================
# 4. TRAIN FINAL MODEL
# =============================================================================

def train_final(model_type: str, params: dict, X: np.ndarray, y: np.ndarray):
    if model_type == "xgboost":
        import xgboost as xgb
        clf = xgb.XGBClassifier(
            **params, use_label_encoder=False,
            eval_metric="logloss", random_state=42,
        )
    else:
        import lightgbm as lgb
        clf = lgb.LGBMClassifier(**params, random_state=42, verbose=-1)
    clf.fit(X, y)
    return clf


# =============================================================================
# 5. PROBABILITY CALIBRATION
# =============================================================================

def calibrate_model(clf, X_calib: np.ndarray, y_calib: np.ndarray, method: str = "sigmoid"):
    """
    Wrap an already-fitted classifier with a held-out probability-calibration
    layer (Platt/sigmoid scaling by default).

    Raw GBM probabilities — especially under a tuned scale_pos_weight — are
    not well-calibrated, and that's a likely driver of a threshold picked on
    Val not transferring cleanly to Test/OOT. This fits the calibrator on
    X_calib/y_calib (a slice the model was NOT trained on), and returns a
    wrapper whose .predict_proba() gives calibrated probabilities.

    NOTE: the returned object is a CalibratedClassifierCV wrapper, not the
    raw xgboost/lightgbm estimator — it does NOT have .save_model() /
    .booster_, and is not what SHAP's TreeExplainer expects. Keep a separate
    reference to the raw fitted model for SHAP and for native-format saving.
    """
    from sklearn.calibration import CalibratedClassifierCV
    try:
        # sklearn >= 1.6: cv="prefit" is deprecated in favor of FrozenEstimator
        from sklearn.frozen import FrozenEstimator
        calibrated = CalibratedClassifierCV(FrozenEstimator(clf), method=method)
    except ImportError:
        calibrated = CalibratedClassifierCV(clf, method=method, cv="prefit")
    calibrated.fit(X_calib, y_calib)
    return calibrated


# =============================================================================
# 6. THRESHOLD CALIBRATION
# =============================================================================

def select_threshold(clf, X: np.ndarray, y: np.ndarray,
                      recall_target: float = RECALL_TARGET):
    """
    Pick the operating threshold off the Val precision-recall curve:
    the threshold that MAXIMIZES PRECISION subject to recall >= recall_target.
    Falls back to the threshold with recall closest to the target if the
    target isn't reachable at all (e.g. a very weak model).

    This replaces the implicit "use predict()'s default 0.5" choice with an
    explicit, inspectable decision tied to the recall floor we actually care
    about for this use case.
    """
    from sklearn.metrics import precision_recall_curve

    y_prob = clf.predict_proba(X)[:, 1]
    prec, rec, thr = precision_recall_curve(y, y_prob)
    if len(thr) == 0:
        return 0.5, float(prec[-1]), float(rec[-1])

    # precision_recall_curve appends a final (precision=1, recall=0) point
    # with no corresponding threshold — drop it so arrays line up with thr.
    prec, rec = prec[:-1], rec[:-1]

    meets_target = rec >= recall_target
    if meets_target.any():
        cand = np.where(meets_target)[0]
        idx  = cand[np.argmax(prec[cand])]
    else:
        idx = int(np.argmin(np.abs(rec - recall_target)))

    return float(thr[idx]), float(prec[idx]), float(rec[idx])


# =============================================================================
# 7. METRICS
# =============================================================================

def compute_metrics(clf, X, y, split_name: str, threshold: float = 0.5) -> dict:
    from sklearn.metrics import (
        recall_score, precision_score, f1_score,
        roc_auc_score, average_precision_score, confusion_matrix,
    )
    y_prob = clf.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    cm     = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    m = dict(
        split     = split_name,
        threshold = round(threshold, 4),
        n_rows    = len(y),
        n_pos     = int(y.sum()),
        recall    = round(recall_score(y, y_pred, zero_division=0), 4),
        precision = round(precision_score(y, y_pred, zero_division=0), 4),
        f1        = round(f1_score(y, y_pred, zero_division=0), 4),
        auc_roc   = round(roc_auc_score(y, y_prob), 4),
        pr_auc    = round(average_precision_score(y, y_prob), 4),  # threshold-independent
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
    )
    print(f"  [{split_name}]  thr={threshold:.3f}  recall={m['recall']:.4f}  "
          f"prec={m['precision']:.4f}  f1={m['f1']:.4f}  "
          f"auc={m['auc_roc']:.4f}  pr_auc={m['pr_auc']:.4f}  "
          f"TP={tp} FP={fp} TN={tn} FN={fn}")
    return m


# =============================================================================
# 8. PSI
# =============================================================================

def psi(expected, actual, bins=10) -> float:
    bp    = np.linspace(0, 1, bins + 1)
    e_pct = np.histogram(expected, bins=bp)[0] / len(expected)
    a_pct = np.histogram(actual,   bins=bp)[0] / len(actual)
    e_pct = np.where(e_pct == 0, 1e-4, e_pct)
    a_pct = np.where(a_pct == 0, 1e-4, a_pct)
    return round(float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))), 4)


# =============================================================================
# 9. SHAP
# =============================================================================

def top_shap(clf, X: np.ndarray, features: list, n: int = 8) -> dict:
    try:
        import shap
        explainer   = shap.TreeExplainer(clf)
        idx         = np.random.choice(len(X), min(500, len(X)), replace=False)
        shap_vals   = explainer.shap_values(X[idx])
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        mean_abs    = np.abs(shap_vals).mean(axis=0)
        ranked      = sorted(zip(features, mean_abs), key=lambda x: x[1], reverse=True)
        print("  SHAP top features:")
        for f, v in ranked[:n]:
            print(f"    {f:<35} {v:.4f}")
        return {f: round(float(v), 4) for f, v in ranked}
    except Exception as e:
        print(f"  [warn] SHAP skipped: {e}")
        return {}


# =============================================================================
# 10. FEATURE ~ LABEL CORRELATION (cross-check against SHAP)
# =============================================================================

def feature_label_correlation(df: pd.DataFrame, features: list, label: str) -> dict:
    """
    Point-biserial correlation between each raw feature and the binary label
    (== Pearson r against a 0/1 target). A cheap, model-free cross-check
    against SHAP:
      - feature ranks LOW on both SHAP and raw correlation -> consistent,
        the feature plausibly carries little signal in this dataset.
      - feature ranks LOW on SHAP but has a real raw correlation -> worth
        investigating: the model may not be using available signal
        (crowded out by a collinear/dominant feature), or there's a
        feature-engineering bug (wrong lag window, scaling, leakage in a
        DIFFERENT feature that's absorbing its signal, etc).
    """
    corrs = {}
    y = df[label].values.astype(float)
    for f in features:
        x = df[f].values.astype(float)
        corrs[f] = 0.0 if np.std(x) == 0 else float(np.corrcoef(x, y)[0, 1])
    ranked = dict(sorted(corrs.items(), key=lambda kv: abs(kv[1]), reverse=True))
    print("  Feature ~ label correlation (train set, |r| descending):")
    for f, r in ranked.items():
        print(f"    {f:<35} {r:+.4f}")
    return ranked


# =============================================================================
# 11. THRESHOLD SENSITIVITY
# =============================================================================

def threshold_sensitivity(clf, X: np.ndarray, y: np.ndarray, threshold: float,
                           deltas=(-0.05, -0.02, 0.0, 0.02, 0.05)) -> list:
    """
    Recall/precision at the calibrated threshold +/- small deltas, evaluated
    on a split the threshold was NOT chosen on (Test, here — not Val-thresh,
    which would just trivially confirm the point it was picked at).

    A flat profile means the operating point sits on a plateau and should
    transfer reliably to OOT/production. A profile that swings sharply means
    the threshold sits on a steep part of the curve — exactly what made
    XGBoost's threshold land at recall=0.711 on Val but recall=0.885 on Test
    in the prior run, while LightGBM's held much closer (0.711 -> 0.757).
    """
    from sklearn.metrics import precision_score, recall_score
    y_prob = clf.predict_proba(X)[:, 1]
    rows = []
    for d in deltas:
        t = float(np.clip(threshold + d, 0.0, 1.0))
        y_pred = (y_prob >= t).astype(int)
        rows.append(dict(
            delta=round(d, 4), threshold=round(t, 4),
            recall=round(recall_score(y, y_pred, zero_division=0), 4),
            precision=round(precision_score(y, y_pred, zero_division=0), 4),
        ))
    return rows


# =============================================================================
# 12. PLOTS
# =============================================================================

def plot_optuna_history(histories: dict, out_dir: Path):
    """
    Left  — trial-by-trial CV F2-score + running best
    Right — F2-score distribution across all trials
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Optuna HPT — {N_OPTUNA_TRIALS} Trials, {N_CV_FOLDS}-Fold Time-Series CV "
        f"(objective: F{F2_BETA:.0f}-score)",
        fontsize=13, fontweight="bold",
    )

    ax = axes[0]
    for mt, vals in histories.items():
        c = CLR[mt]
        ax.scatter(range(len(vals)), vals, alpha=0.2, color=c, s=8)
        ax.plot(
            range(len(vals)), pd.Series(vals).cummax().values,
            color=c, linewidth=2.2, label=f"{mt}  (best={max(vals):.4f})",
        )
    ax.set(xlabel="Trial #", ylabel=f"Mean CV F{F2_BETA:.0f}-score ({N_CV_FOLDS} folds)",
           title="Trial History — Running Best", ylim=(0, 1.05))
    ax.legend(fontsize=9); ax.grid(alpha=0.25)

    ax = axes[1]
    bins = np.linspace(0, 1, 30)
    for mt, vals in histories.items():
        c = CLR[mt]
        ax.hist(vals, bins=bins, alpha=0.5, color=c, label=mt,
                edgecolor="white", linewidth=0.4)
        ax.axvline(np.median(vals), color=c, linestyle=":", linewidth=1.8,
                   label=f"{mt} median ({np.median(vals):.4f})")
    ax.set(xlabel=f"Mean CV F{F2_BETA:.0f}-score", ylabel="Count",
           title="F2-score Distribution Across All Trials")
    ax.legend(fontsize=9); ax.grid(alpha=0.25)

    plt.tight_layout()
    p = out_dir / "optuna_history.png"
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [plot] {p.name}")


def plot_pr_and_threshold(models: dict, X_val, y_val, thresholds: dict, out_dir: Path):
    """
    Left  — Precision-Recall curve (AP labelled); dots = default 0.5,
            stars = calibrated operating threshold actually used for eval.
    Right — Threshold vs Recall / Precision sweep, with the calibrated
            threshold marked per model.
    """
    from sklearn.metrics import (
        precision_recall_curve, average_precision_score,
        precision_score, recall_score,
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Model Comparison — Validation Set (Jan–Jun 2019)",
                 fontsize=13, fontweight="bold")

    probs      = {mt: m.predict_proba(X_val)[:, 1] for mt, m in models.items()}
    thresh_sweep = np.linspace(0.01, 0.99, 200)
    baseline   = float(y_val.mean())

    # ── PR curve ─────────────────────────────────────────────────────────────
    ax = axes[0]
    for mt, yp in probs.items():
        prec, rec, thr = precision_recall_curve(y_val, yp)
        ap = average_precision_score(y_val, yp)
        ax.plot(rec, prec, color=CLR[mt], linewidth=2.2,
                label=f"{mt}  (AP={ap:.3f})")
        # mark default 0.5 threshold on curve
        di = np.abs(thr - 0.5).argmin() if len(thr) else 0
        ax.scatter(rec[di], prec[di], color=CLR[mt], s=70, zorder=5,
                   marker="o", edgecolors="white", linewidths=0.8)
        # mark calibrated operating threshold
        cal_t = thresholds.get(mt)
        if cal_t is not None and len(thr):
            ci = np.abs(thr - cal_t).argmin()
            ax.scatter(rec[ci], prec[ci], color=CLR[mt], s=160, zorder=6,
                       marker="*", edgecolors="black", linewidths=0.8,
                       label=f"{mt} calibrated (thr={cal_t:.3f})")
    ax.axhline(baseline, color="gray", linestyle="--", alpha=0.7,
               label=f"No-skill baseline ({baseline:.3f})")
    ax.set(xlabel="Recall", ylabel="Precision",
           title="Precision-Recall Curve", xlim=(0, 1.02), ylim=(0, 1.05))
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.text(0.02, 0.04, "● default (0.5)   ★ calibrated threshold",
            transform=ax.transAxes, fontsize=8, color="gray")

    # ── threshold sweep ───────────────────────────────────────────────────────
    ax = axes[1]
    for mt, yp in probs.items():
        c = CLR[mt]
        p_vals, r_vals = [], []
        for t in thresh_sweep:
            ypred = (yp >= t).astype(int)
            if ypred.sum() == 0:
                p_vals.append(1.0); r_vals.append(0.0)
            else:
                p_vals.append(precision_score(y_val, ypred, zero_division=0))
                r_vals.append(recall_score(y_val, ypred, zero_division=0))
        ax.plot(thresh_sweep, r_vals, color=c, linewidth=2.2, linestyle="-",
                label=f"{mt} recall")
        ax.plot(thresh_sweep, p_vals, color=c, linewidth=2.2, linestyle="--",
                label=f"{mt} precision")
        cal_t = thresholds.get(mt)
        if cal_t is not None:
            ax.axvline(cal_t, color=c, linestyle=":", linewidth=1.6, alpha=0.8)

    ax.axhline(RECALL_TARGET, color="red", linestyle="--", linewidth=1.4, alpha=0.85,
               label=f"Recall target ({RECALL_TARGET})")
    ax.set(xlabel="Decision Threshold", ylabel="Score",
           title="Threshold vs Recall / Precision\n(dotted vertical = calibrated threshold per model)",
           xlim=(0, 1), ylim=(0, 1.05))
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.25)
    ax.text(0.02, 0.04, "Solid = recall  |  Dashed = precision",
            transform=ax.transAxes, fontsize=8, color="gray")

    plt.tight_layout()
    p = out_dir / "model_comparison.png"
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [plot] {p.name}")


def plot_eval_table(results: list, out_dir: Path):
    """
    Renders the evaluation metrics as a styled table figure —
    one row per (model × split) combination, ready for the presentation.
    """
    rows, hi_val, hi_oot = [], {}, {}

    for r in results:
        rows.append([
            r["model_type"].upper(),
            r["split"],
            f"{r['threshold']:.3f}",
            f"{r['recall']:.3f}",
            f"{r['precision']:.3f}",
            f"{r['f1']:.3f}",
            f"{r['auc_roc']:.3f}",
            f"{r['pr_auc']:.3f}",
            f"{r['tp']} / {r['fp']} / {r['tn']} / {r['fn']}",
        ])

    cols = ["Model", "Split", "Thr", "Recall", "Precision", "F1", "AUC-ROC", "PR-AUC", "TP/FP/TN/FN"]

    fig, ax = plt.subplots(figsize=(14.5, 0.55 * len(rows) + 1.4))
    ax.axis("off")
    tbl = ax.table(
        cellText=rows, colLabels=cols,
        cellLoc="center", loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.55)

    # header styling
    for j in range(len(cols)):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    # row colours by split
    split_colours = {
        "Val":  "#fef9e7",
        "Test": "#fde8d8",
        "OOT":  "#d6eaf8",
    }
    for i, row in enumerate(rows, start=1):
        bg = split_colours.get(row[1], "white")
        for j in range(len(cols)):
            tbl[i, j].set_facecolor(bg)

    ax.set_title("Evaluation Results — Val / Test / OOT",
                 fontsize=12, fontweight="bold", pad=12)
    plt.tight_layout()
    p = out_dir / "eval_table.png"
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [plot] {p.name}")


# =============================================================================
# 13. PROMOTION GATE (no MLflow — console only)
# =============================================================================

def check_gate(test_m: dict, oot_m: dict) -> bool:
    oot_drop = test_m["recall"] - oot_m["recall"]
    gates = {
        f"recall_test >= {RECALL_TARGET}":   test_m["recall"] >= RECALL_TARGET,
        f"oot_drop   <= 10pp":               oot_drop <= 0.10,
        f"auc_roc    >= 0.75 (test)":        test_m["auc_roc"] >= 0.75,
    }
    print("\n[gate] Promotion gate")
    for desc, passed in gates.items():
        print(f"  {'PASS' if passed else 'FAIL'}  {desc}")
    all_pass = all(gates.values())
    print(f"  → {'PROMOTE' if all_pass else 'HOLD'}")
    return all_pass


# =============================================================================
# MAIN
# =============================================================================

def main():
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print("Dengue Outbreak Risk Prediction — Train + Evaluate")
    print("=" * 60)

    # ── 1. data ───────────────────────────────────────────────────────────────
    splits  = load_and_split(INPUT_PATH)
    X_train_raw = splits["train"][FEATURES].values
    y_train_raw = splits["train"][LABEL].values
    X_val        = splits["val"][FEATURES].values;        y_val        = splits["val"][LABEL].values
    X_val_calib  = splits["val_calib"][FEATURES].values;   y_val_calib  = splits["val_calib"][LABEL].values
    X_val_thresh = splits["val_thresh"][FEATURES].values;  y_val_thresh = splits["val_thresh"][LABEL].values
    X_test = splits["test"][FEATURES].values;  y_test = splits["test"][LABEL].values
    X_oot  = splits["oot"][FEATURES].values;   y_oot  = splits["oot"][LABEL].values

    # ── 2. train data (no resampling — imbalance handled via scale_pos_weight) ─
    X_train, y_train = X_train_raw, y_train_raw

    # ── 3. Optuna — tune both models ──────────────────────────────────────────
    histories              = {}
    raw_models              = {}   # model_type -> raw fitted xgb/lgb estimator
    trained_models          = {}   # model_type -> calibrated wrapper (used for eval/plots)
    calibrated_thresholds   = {}   # model_type -> threshold chosen on Val-thresh
    val_thresh_metrics      = {}   # model_type -> (precision, recall) at that threshold
    sensitivity_results     = {}   # model_type -> threshold sensitivity rows (on Test)
    best_run = {"precision": -1, "model": None, "model_type": None, "threshold": None}

    for mt in ["xgboost", "lightgbm"]:
        print(f"\n[optuna] {mt} — {N_OPTUNA_TRIALS} trials")
        study = optuna.create_study(direction="maximize")
        study.optimize(
            make_objective(splits["train"], mt),
            n_trials=N_OPTUNA_TRIALS,
            show_progress_bar=True,
        )
        histories[mt] = [
            t.value if t.value is not None else 0.0 for t in study.trials
        ]
        print(f"  best CV F{F2_BETA:.0f}-score: {study.best_value:.4f}")

        # ── 4. train final model on full (raw, imbalanced) train set ──────────
        clf = train_final(mt, study.best_params, X_train, y_train)
        raw_models[mt] = clf

        # ── 5. calibrate probabilities on Val-calib (Jan-Mar 2019) ────────────
        cal_clf = calibrate_model(clf, X_val_calib, y_val_calib, method="sigmoid")
        trained_models[mt] = cal_clf

        # ── 6. pick threshold on Val-thresh (Apr-Jun 2019) — disjoint from the
        #      slice used to fit calibration above ─────────────────────────────
        thr, val_prec, val_rec = select_threshold(cal_clf, X_val_thresh, y_val_thresh, RECALL_TARGET)
        calibrated_thresholds[mt] = thr
        val_thresh_metrics[mt]    = (val_prec, val_rec)
        print(f"  calibrated threshold: {thr:.3f}  "
              f"(val-thresh recall={val_rec:.4f}, precision={val_prec:.4f})  "
              f"({'PASS' if val_rec >= RECALL_TARGET else 'FAIL'} recall target)")

        # ── threshold sensitivity, evaluated on Test (not on the slice the ────
        #    threshold was picked on) — flags a threshold sitting on a cliff
        sens = threshold_sensitivity(cal_clf, X_test, y_test, thr)
        sensitivity_results[mt] = sens
        print(f"  threshold sensitivity ({mt}, Test, Δ around {thr:.3f}):")
        for row in sens:
            marker = "  <-- chosen" if row["delta"] == 0.0 else ""
            print(f"    Δ={row['delta']:+.2f}  thr={row['threshold']:.3f}  "
                  f"recall={row['recall']:.4f}  precision={row['precision']:.4f}{marker}")

        # select best model by precision at the recall floor — i.e. whichever
        # model gives the most USABLE alerts once both hit the same recall bar
        if val_prec > best_run["precision"]:
            best_run = {"precision": val_prec, "model": cal_clf,
                        "model_type": mt, "threshold": thr}

    # ── 7. full evaluation on val / test / OOT (calibrated probabilities) ────
    print("\n[eval] Full metrics")
    all_results = []
    best_clf    = best_run["model"]        # calibrated wrapper
    best_mt     = best_run["model_type"]
    best_raw    = raw_models[best_mt]      # raw estimator — for SHAP / native save

    for Xs, ys, name in [
        (X_val,  y_val,  "Val"),
        (X_test, y_test, "Test"),
        (X_oot,  y_oot,  "OOT"),
    ]:
        for mt, clf in trained_models.items():
            m = compute_metrics(clf, Xs, ys, name, threshold=calibrated_thresholds[mt])
            m["model_type"] = mt
            all_results.append(m)

    # PSI — score distribution shift (best model, test → OOT), calibrated probs
    test_scores = best_clf.predict_proba(X_test)[:, 1]
    oot_scores  = best_clf.predict_proba(X_oot)[:, 1]
    psi_val     = psi(test_scores, oot_scores)
    psi_flag    = ("stable" if psi_val < 0.1
                   else "minor shift" if psi_val < 0.2
                   else "significant drift")
    print(f"\n[psi] {best_mt} — test→OOT: {psi_val:.4f} ({psi_flag})")

    # SHAP — needs the RAW tree model, not the CalibratedClassifierCV wrapper
    print(f"\n[shap] {best_mt}")
    shap_dict = top_shap(best_raw, X_test, FEATURES)

    # Feature ~ label correlation — model-free cross-check against SHAP above
    print(f"\n[corr] Feature ~ label correlation (diagnostic cross-check vs SHAP)")
    corr_dict = feature_label_correlation(splits["train"], FEATURES, LABEL)

    # ── 8. promotion gate ─────────────────────────────────────────────────────
    best_test_m = next(r for r in all_results
                       if r["split"] == "Test" and r["model_type"] == best_mt)
    best_oot_m  = next(r for r in all_results
                       if r["split"] == "OOT"  and r["model_type"] == best_mt)
    promoted = check_gate(best_test_m, best_oot_m)

    # ── 9. plots ──────────────────────────────────────────────────────────────
    print("\n[plot] Generating plots")
    plot_optuna_history(histories, OUT_DIR)
    plot_pr_and_threshold(trained_models, X_val, y_val, calibrated_thresholds, OUT_DIR)
    plot_eval_table(all_results, OUT_DIR)

    # ── 10. save best model ───────────────────────────────────────────────────
    # native format = raw booster (no calibration layer, but portable/inspectable)
    if best_mt == "xgboost":
        best_raw.save_model(OUT_DIR / "best_model.xgb")
    else:
        best_raw.booster_.save_model(str(OUT_DIR / "best_model.lgb"))

    # the calibrated wrapper is the actual scoring artifact matched to
    # calibrated_threshold above — native xgb/lgb formats don't carry the
    # calibration layer, so persist it separately via joblib
    import joblib
    joblib.dump(best_clf, OUT_DIR / "best_model_calibrated.joblib")

    meta = dict(
        model_type                = best_mt,
        calibrated_threshold      = round(best_run["threshold"], 4),
        val_precision_at_thr      = round(best_run["precision"], 4),
        val_recall_at_thr         = round(val_thresh_metrics[best_mt][1], 4),
        recall_target              = RECALL_TARGET,
        f2_beta                    = F2_BETA,
        calibration_method         = "sigmoid",
        all_calibrated_thresholds  = {mt: round(t, 4) for mt, t in calibrated_thresholds.items()},
        threshold_sensitivity_best = sensitivity_results[best_mt],
        psi                        = psi_val,
        psi_flag                   = psi_flag,
        promoted                   = promoted,
        shap_top_features          = dict(list(shap_dict.items())[:8]),
        feature_label_correlation  = {f: round(r, 4) for f, r in corr_dict.items()},
        n_optuna_trials            = N_OPTUNA_TRIALS,
        n_cv_folds                 = N_CV_FOLDS,
        note_oot=(
            "OOT (Jan-Nov 2020) covers DENV-3 serotype shift outbreak "
            "(35,000+ cases). Recall drop expected and treated as concept "
            "drift case study, not model failure."
        ),
    )
    with open(OUT_DIR / "run_summary.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n[done] Best model : {best_mt}  "
          f"(threshold={best_run['threshold']:.3f}, "
          f"val precision={best_run['precision']:.4f}, "
          f"val recall={val_thresh_metrics[best_mt][1]:.4f})")
    print(f"       Outputs     : {OUT_DIR}")
    print(f"       Promoted    : {promoted}")


if __name__ == "__main__":
    main()

Dengue Outbreak Risk Prediction — Train + Evaluate
[data] Loading subzone_features.parquet
  train: 51,238 rows | positive: 5,337 (10.4%)
  val  :  4,658 rows | positive: 779 (16.7%)
  val_calib:  1,096 rows | positive: 127 (11.6%)
  val_thresh:  3,562 rows | positive: 652 (18.3%)
  test :  6,850 rows | positive: 2,003 (29.2%)
  oot  :  7,398 rows | positive: 1,901 (25.7%)

[optuna] xgboost — 150 trials


  0%|          | 0/150 [00:00<?, ?it/s]

  best CV F2-score: 0.4253
  calibrated threshold: 0.148  (val-thresh recall=0.7285, precision=0.4419)  (PASS recall target)
  threshold sensitivity (xgboost, Test, Δ around 0.148):
    Δ=-0.05  thr=0.098  recall=0.9406  precision=0.4567
    Δ=-0.02  thr=0.128  recall=0.9406  precision=0.4595
    Δ=+0.00  thr=0.148  recall=0.9406  precision=0.4623  <-- chosen
    Δ=+0.02  thr=0.168  recall=0.9386  precision=0.4671
    Δ=+0.05  thr=0.198  recall=0.9366  precision=0.4690

[optuna] lightgbm — 150 trials


  0%|          | 0/150 [00:00<?, ?it/s]

  best CV F2-score: 0.4081
  calibrated threshold: 0.190  (val-thresh recall=0.7040, precision=0.4632)  (PASS recall target)
  threshold sensitivity (lightgbm, Test, Δ around 0.190):
    Δ=-0.05  thr=0.140  recall=0.9256  precision=0.4617
    Δ=-0.02  thr=0.170  recall=0.9156  precision=0.4707
    Δ=+0.00  thr=0.190  recall=0.8982  precision=0.4898  <-- chosen
    Δ=+0.02  thr=0.210  recall=0.8572  precision=0.5277
    Δ=+0.05  thr=0.240  recall=0.8008  precision=0.5508

[eval] Full metrics
  [Val]  thr=0.148  recall=0.7240  prec=0.4150  f1=0.5276  auc=0.8433  pr_auc=0.5646  TP=564 FP=795 TN=3084 FN=215
  [Val]  thr=0.190  recall=0.7009  prec=0.4379  f1=0.5390  auc=0.8481  pr_auc=0.5719  TP=546 FP=701 TN=3178 FN=233
  [Test]  thr=0.148  recall=0.9406  prec=0.4623  f1=0.6199  auc=0.8450  pr_auc=0.6573  TP=1884 FP=2191 TN=2656 FN=119
  [Test]  thr=0.190  recall=0.8982  prec=0.4898  f1=0.6339  auc=0.8463  pr_auc=0.6665  TP=1799 FP=1874 TN=2973 FN=204
  [OOT]  thr=0.148  recall=0.9006  pre